# **Portfolio Forecast Demo: Choosing a Simulator**

This demo builds a synthetic hourly history whose model is known, then runs the simulators on it to show what each
setting does and whether the fits recover the truth. Everything is seeded and generated here, so it needs no network.

1. Build a two-regime hourly history on real exchange labels, with the true parameters stated.
2. Fit the parametric simulator to Student's t returns and see which family BIC picks.
3. Let the regime-switching simulator choose its regime count, and compare its estimates with the truth.
4. Compare `gap_model="regime"` with `"independent"` on the same history.
5. Compare `kappa` 0 with a larger value through the expected regime durations.
6. Compare the tails of `bootstrap=True` and `bootstrap=False`.
7. Trigger the minimum sample check on a short history and read its message.

## **Imports**

In [1]:
# The two fitted simulators and the future-mark layout, plus numpy and pandas for the synthetic history
import warnings

import numpy as np
import pandas as pd

from portfolio_forecast.forecast import parametric_monte_carlo, regime_switching_monte_carlo
from portfolio_forecast.utils import next_trading_dates

In [2]:
# On intraday history every simulator warns that sim_length counts sessions; this demo knows, so silence it
warnings.filterwarnings('ignore', message='.*counts sessions', category=UserWarning)
pd.set_option('display.float_format', '{:.6g}'.format)
pd.set_option('display.width', 120)

## **Synthetic History**

### **Hourly Labels**

The labels come from the package's own calendar tools, so they have everything real XNYS hourly data has: an open
mark at 9:30 that ends each overnight gap, hourly bar closes at 10:30, ..., 15:30, a short last bar from 15:30 to
the 16:00 close, and half-day sessions that close at 13:00 with a short 12:30 to 13:00 bar. One session of hourly
marks goes into [`next_trading_dates`](../README.md#next_trading_dates), which continues it for 300 sessions with
each mark's class and length ratio $\rho = \tau/\Delta$. The history starts at the first of those opens, as a
backtest's value series does.

In [3]:
# One session of hourly marks (bars starting 9:30, ..., 15:30, labelled by their closes), continued 300 sessions
seed_marks = pd.DatetimeIndex(
    list(pd.date_range('2023-12-29 09:30', '2023-12-29 15:30', freq='h', tz='America/New_York'))
    + [pd.Timestamp('2023-12-29 16:00', tz='America/New_York')]
)
layout = next_trading_dates(seed_marks, 300, interval='1h')

# The history starts at the first open; each later mark ends one return, of the class and rho the layout gives
index = layout.dates
classes = layout.classes[1:]
rho = layout.tau[1:]
is_gap = classes == 'gap'
n_returns = len(classes)
print(f'{len(index)} marks from {index[0]} to {index[-1]}')
print(f'{n_returns} returns: {is_gap.sum()} gaps and {(~is_gap).sum()} regular bars, '
      f'{(rho < 1).sum()} of them short (rho = {np.unique(rho[rho < 1])})')
print('Half-day closes:', [str(t) for t in index[(index.hour == 13) & (index.minute == 0)]])

2391 marks from 2024-01-02 09:30:00-05:00 to 2025-03-13 16:00:00-04:00
2390 returns: 299 gaps and 2091 regular bars, 300 of them short (rho = [0.5])
Half-day closes: ['2024-07-03 13:00:00-04:00', '2024-11-29 13:00:00-05:00', '2024-12-24 13:00:00-05:00']


### **Two-Regime Returns**

A hidden chain with two regimes, calm (0) and turbulent (1), steps once per mark, gaps included, by the transition
matrix $P$. Given regime $s$, a gap is $\mathcal N(\mu_{s,\text{gap}}, \sigma^2_{s,\text{gap}})$ and a regular bar
of length ratio $\rho$ is $\mathcal N(\rho\,\mu_{s,\text{reg}},\ \rho\,\sigma^2_{s,\text{reg}})$: exactly the model
of [`regime_switching_monte_carlo`](../README.md#regime-switching-regime_switching_monte_carlo) with
`gap_model="regime"`, short bars scaled as in [Short bars](../README.md#what-all-three-share). The truth:

In [4]:
# The true model: transition matrix, then per-regime means and volatilities per gap and per full bar
P = np.array([[0.98, 0.02],
              [0.04, 0.96]])
mu_reg = np.array([0.0003, -0.0006])
sigma_reg = np.array([0.002, 0.007])
mu_gap = np.array([0.001, -0.002])
sigma_gap = np.array([0.004, 0.015])

truth = pd.DataFrame({'Mean': mu_reg, 'Volatility': sigma_reg, 'Gap Mean': mu_gap, 'Gap Volatility': sigma_gap,
                      'Expected Duration': 1 / (1 - np.diag(P))}, index=pd.Index([0, 1], name='Regime'))
truth

,Mean,Volatility,Gap Mean,Gap Volatility,Expected Duration
Regime,,,,,
0,0.0003,0.002,0.001,0.004,50
1,-0.0006,0.007,-0.002,0.015,25


In [5]:
# Run the chain from the calm regime, then draw each return from its regime's normal for its class and rho
rng = np.random.default_rng(42)
regime = np.empty(n_returns, dtype=int)
regime[0] = 0
for t in range(1, n_returns):
    regime[t] = rng.choice(2, p=P[regime[t - 1]])
z = rng.standard_normal(n_returns)
ell = np.where(is_gap, mu_gap[regime] + sigma_gap[regime] * z,
               rho * mu_reg[regime] + np.sqrt(rho) * sigma_reg[regime] * z)
values = pd.Series(np.exp(np.r_[0.0, np.cumsum(ell)]), index=index)

# What the sample itself holds: marks and gaps per regime, and the transition frequencies the chain realized
counts = np.zeros((2, 2))
np.add.at(counts, (regime[:-1], regime[1:]), 1)
realized_P = counts / counts.sum(axis=1, keepdims=True)
print('Marks per regime:', np.bincount(regime), ' gaps per regime:', np.bincount(regime[is_gap]))
print('Realized transition frequencies:\n', realized_P.round(4))
print('Realized expected durations:', (1 / (1 - np.diag(realized_P))).round(1))

Marks per regime: [1424  966]  gaps per regime: [176 123]
Realized transition frequencies:
 [[0.9733 0.0267]
 [0.0383 0.9617]]
Realized expected durations: [37.5 26.1]


## **Parametric: Picking a Family**

[Parametric family selection](../README.md#mc5-parametric-family-selection) fits the normal, Student's t and Johnson SU to each class and
keeps the lowest BIC. To give it something to find, this section uses a second history on the same labels whose
regular returns are i.i.d. Student's t with 4 degrees of freedom per full bar (short bars scaled by $\rho$ as above)
and whose gaps are normal. `verbose=True` prints every candidate's score, and `fit` holds the result; `return_fit`
is no longer needed.

In [6]:
# i.i.d. history: regular bars m + scale * t_4 per full bar, shrunk to short bars by rho; gaps N(0.0005, 0.006^2)
rng = np.random.default_rng(42)
m, scale = 0.0002, 0.002
full_bar = m + scale * rng.standard_t(4, n_returns)
ell_t = np.where(is_gap, rng.normal(0.0005, 0.006, n_returns), rho * m + np.sqrt(rho) * (full_bar - m))
values_t = pd.Series(np.exp(np.r_[0.0, np.cumsum(ell_t)]), index=index)

parametric = parametric_monte_carlo(values_t, sim_length=20, n_sims=2000, random_state=np.random.default_rng(42))

regular       norm BIC: -18,711.20


regular          t BIC: -18,960.59


regular  johnsonsu BIC: -18,956.37
regular Selected: t  (df 3.98, loc 0.00019, scale 0.00198)
gap       norm BIC: -2,159.66
gap          t BIC: -2,154.95


gap  johnsonsu BIC: -2,150.81
gap Selected: norm  (loc -0.000192, scale 0.00641)


In [7]:
# The regular class's fit: the family, its parameters per full bar (true df 4, loc 0.0002, scale 0.002), the scores
parametric.fit['regular']

{'Distribution': 't',
 'Parameters': {'df': 3.9781967997014984,
  'loc': 0.00018972468060756955,
  'scale': 0.0019840830472041054},
 'Criterion': 'BIC',
 'BIC': -18960.59442716537,
 'Candidates': {'t': -18960.59442716537,
  'johnsonsu': -18956.368056797677,
  'norm': -18711.200323817433}}

BIC picks t for the regular bars: the normal can't produce the tails, and Johnson SU's extra skew parameter doesn't
earn its $\ln n$ penalty on symmetric data. The gaps, which are normal, stay normal.

## **Regime Switching: Picking the Regime Count**

With `n_regimes=None` the simulator fits 1, 2 and 3 regimes and keeps the lowest BIC among the counts that pass
the [minimum sample](../README.md#mc6-minimum-sample) ([regime-switching rules](../README.md#mc7-regime-switching-hmm),
[`criterion`](../README.md#criterion-both-simulators)). Three EM starts per count are enough here and keep the demo
fast; the default is 10.

In [8]:
# Let BIC choose the regime count on the two-regime history; verbose prints every count's score and the regimes
regimes = regime_switching_monte_carlo(values, n_starts=3, sim_length=20, n_sims=2000,
                                       random_state=np.random.default_rng(42))

1 regime(s) BIC: -18,593.64
2 regime(s) BIC: -19,828.02


3 regime(s) BIC: -19,765.80
Selected: 2 regime(s)
Regime 0: regular mean +0.0353%, vol 0.2000% | gap mean +0.1101%, vol 0.4070% | expected duration 40.1 marks
Regime 1: regular mean -0.0557%, vol 0.6873% | gap mean -0.2369%, vol 1.4906% | expected duration 28.0 marks


In [9]:
# Truth against estimate; the fit orders regimes by volatility, so its regime 0 is the calm one, as in the truth
fitted = regimes.fit['Regimes']
fitted_P = regimes.fit['Transition Matrix'].to_numpy()
rows = {
    'P[0, 0]': (P[0, 0], realized_P[0, 0], fitted_P[0, 0]),
    'P[0, 1]': (P[0, 1], realized_P[0, 1], fitted_P[0, 1]),
    'P[1, 0]': (P[1, 0], realized_P[1, 0], fitted_P[1, 0]),
    'P[1, 1]': (P[1, 1], realized_P[1, 1], fitted_P[1, 1]),
}
for s in (0, 1):
    for column, true in (('Mean', mu_reg), ('Volatility', sigma_reg), ('Gap Mean', mu_gap),
                         ('Gap Volatility', sigma_gap)):
        sample = ell[(regime == s) & is_gap] if column.startswith('Gap') else None
        if sample is None:
            # Regular sample moments per full bar, by the simulator's own rho-weighted formulas
            bars = (regime == s) & ~is_gap
            mean = ell[bars].sum() / rho[bars].sum()
            realized = mean if column == 'Mean' else np.sqrt(np.mean((ell[bars] - rho[bars] * mean) ** 2
                                                                     / rho[bars]))
        else:
            realized = sample.mean() if column == 'Gap Mean' else sample.std()
        rows[f'{column}, regime {s}'] = (true[s], realized, fitted.loc[s, column])
comparison = pd.DataFrame(rows, index=['Truth', 'Realized in sample', 'Estimate']).T
comparison['Estimate - Truth'] = comparison['Estimate'] - comparison['Truth']
comparison

,Truth,Realized in sample,Estimate,Estimate - Truth
"P[0, 0]",0.98,0.973315,0.975065,-0.00493513
"P[0, 1]",0.02,0.0266854,0.0249351,0.00493513
"P[1, 0]",0.04,0.038342,0.0357095,-0.00429046
"P[1, 1]",0.96,0.961658,0.96429,0.00429046
"Mean, regime 0",0.0003,0.000338296,0.000353393,5.33926e-05
"Volatility, regime 0",0.002,0.00199723,0.0019995,-4.95883e-07
"Gap Mean, regime 0",0.001,0.00105268,0.00110084,0.000100841
"Gap Volatility, regime 0",0.004,0.004048,0.00406954,6.95353e-05
"Mean, regime 1",-0.0006,-0.000539798,-0.000556894,4.31056e-05
"Volatility, regime 1",0.007,0.00689197,0.0068727,-0.000127298


"Realized in sample" is what the drawn regimes actually produced (the sample moments of the returns each regime
emitted, and the chain's transition frequencies), which is the best any estimator could do from this one history.
The volatilities and the transition matrix land close to it. The means are the noisiest numbers: a mean is
estimated with a standard error of about $\sigma/\sqrt{n}$, which for a few hundred gaps is the same size as the
mean itself.

## **Gap Model: Regime or Independent**

[`gap_model`](../README.md#gap_model-regime-switching) decides whether gaps belong to the regime chain. Under
`"regime"` each regime has its own gap normal, so a turbulent regime also has large overnight gaps; under
`"independent"` the chain runs on the bars and every gap comes from one normal. The truth here has regime-dependent
gaps (0.4% and 1.5% volatility), so `"regime"` should fit better. With `bootstrap=False` both likelihoods cover
every return, so their BIC values are comparable. The fan width is the spread between the 5th and 95th percentiles
of the path values, at the first future mark (a gap, since the history ends at a close) and at the horizon.

In [10]:
# The same history and seed under "independent"; the "regime" run is the one from the section above
independent = regime_switching_monte_carlo(values, gap_model='independent', n_starts=3, sim_length=20, n_sims=2000,
                                           random_state=np.random.default_rng(42), verbose=False)


def fan_width(paths, column):
    """Spread between the 5th and 95th percentiles of the path values at one column."""
    low, high = np.quantile(paths[:, column], [0.05, 0.95])
    return high - low




def n_params(S, gap_model):
    """k of an S-regime fit: (S - 1) + S(S - 1), plus 4S under "regime" or 2S + 2 for the gap normal."""
    return (S - 1) + S * (S - 1) + (4 * S if gap_model == 'regime' else 2 * S + 2)


summary = pd.DataFrame({
    name: {'Regime Count': sims.fit['Regime Count'], 'k': n_params(sims.fit['Regime Count'], name),
           'BIC': sims.fit['BIC'][sims.fit['Regime Count']], 'Log Likelihood': sims.fit['Log Likelihood'],
           'Fan Width, First Mark': fan_width(sims.paths, 1), 'Fan Width, Horizon': fan_width(sims.paths, -1)}
    for name, sims in (('regime', regimes), ('independent', independent))
}).T
summary

,Regime Count,k,BIC,Log Likelihood,"Fan Width, First Mark","Fan Width, Horizon"
regime,2,11,-19828,9956.8,0.0478109,0.264784
independent,2,9,-19628,9848.99,0.0333473,0.240309


In [11]:
# How each model draws the next gap: per regime under "regime", one normal under "independent"
print(regimes.fit['Regimes'][['Gap Mean', 'Gap Volatility', 'Current Probability']])
print(independent.fit['Gap'])

          Gap Mean  Gap Volatility  Current Probability
Regime                                                 
0       0.00110084      0.00406954            0.0954669
1      -0.00236887       0.0149055             0.904533
{'Mean': -0.00030326977980425876, 'Volatility': 0.01013253524044503}


Both choose two regimes, but `"regime"` has the higher log likelihood and the lower BIC despite its two extra
parameters, because the gaps really do depend on the regime. The history ends in a likely turbulent regime, so
`"regime"` draws the next gap mostly from the wide turbulent gap normal while `"independent"` uses one normal fitted
to all gaps; compare the first-mark fan widths above. Over the full horizon the chain mixes between regimes and the
two fans end up closer together.

## **Kappa: Regime Durations**

[`kappa`](../README.md#kappa-regime-switching) adds $\kappa$ pretend stays to each row of the transition matrix, so
the expected duration $1/(1 - \hat P_{ii})$ grows. Each unit counts as one observed stay, so its effect depends on
how many marks each regime holds (here about 1,400 and 1,000). The true durations are 50 and 25 marks.

In [12]:
# Two regimes with kappa 0 (maximum likelihood) and 500, on the same history and seed
durations = {}
for kappa in (0, 500):
    sims = regime_switching_monte_carlo(values, n_regimes=2, kappa=kappa, n_starts=3, sim_length=20, n_sims=200,
                                        random_state=np.random.default_rng(42), verbose=False)
    durations[f'kappa = {kappa}'] = list(sims.fit['Regimes']['Expected Duration']) + [sims.fit['Log Likelihood']]
durations['Realized in sample'] = list(1 / (1 - np.diag(realized_P))) + [np.nan]
durations['Truth'] = list(1 / (1 - np.diag(P))) + [np.nan]
pd.DataFrame(durations, index=['Duration, regime 0', 'Duration, regime 1', 'Log Likelihood']).T

,"Duration, regime 0","Duration, regime 1",Log Likelihood
kappa = 0,40.1041,28.0037,9956.8
kappa = 500,61.1834,48.8179,9951.78
Realized in sample,37.4737,26.0811,NaN
Truth,50,25,NaN


At $\kappa = 0$ the durations are close to the ones the chain realized in this sample; the calm regime's falls
short of the true 50 because its runs in this sample happened to be short, which no estimator can see past. A large $\kappa$ stretches both well
beyond the truth at a small cost in log likelihood: the persistence comes from the prior, not the data. Raise
$\kappa$ only when the fitted regimes flicker (durations of a few marks), and choose it by the durations it gives.

## **Bootstrap: Tails**

`bootstrap=False` draws each mark from its regime's fitted normal; `bootstrap=True` resamples the history's own
returns, so the tails are as heavy as the history's and never beyond its largest move
([regime-switching rules](../README.md#mc7-regime-switching-hmm), Simulation). The difference shows on fat-tailed data, so this section
uses the Student's t history with one regime fixed (on i.i.d. data there are no regimes to find, and fixing the
count keeps the comparison about the draws alone). Regular draws are compared per full bar, on full future bars.

In [13]:
# One regime, normal draws against resampled history, on the Student's t history
full_future = next_trading_dates(values_t, 20).tau == 1
tails = {}
for bootstrap in (False, True):
    sims = regime_switching_monte_carlo(values_t, n_regimes=1, bootstrap=bootstrap, n_starts=3, sim_length=20,
                                        n_sims=2000, random_state=np.random.default_rng(42), verbose=False)
    steps = np.diff(np.log(sims.paths), axis=1)
    draws = steps[:, (sims.classes == 'regular') & full_future].ravel()
    tails[f'bootstrap={bootstrap}'] = {
        '0.01% quantile': np.quantile(draws, 0.0001), '99.99% quantile': np.quantile(draws, 0.9999),
        'Largest move': np.abs(draws).max(), 'Final value, 1%': np.quantile(sims.paths[:, -1], 0.01),
        'Final value, 99%': np.quantile(sims.paths[:, -1], 0.99)}

# The history's regular returns per full bar (short bars stretched with the one regime's mean)
bars = ~is_gap
m_hat = ell_t[bars].sum() / rho[bars].sum()
history_full = m_hat + (ell_t[bars] - rho[bars] * m_hat) / np.sqrt(rho[bars])
tails['history'] = {'0.01% quantile': np.quantile(history_full, 0.0001),
                    '99.99% quantile': np.quantile(history_full, 0.9999), 'Largest move': np.abs(history_full).max()}
pd.DataFrame(tails).T

,0.01% quantile,99.99% quantile,Largest move,"Final value, 1%","Final value, 99%"
bootstrap=False,-0.00995078,0.0105029,0.0138932,0.924269,1.12505
bootstrap=True,-0.0141216,0.018256,0.018256,0.922962,1.13425
history,-0.0139746,0.0182376,0.018256,NaN,NaN


The normal's extreme quantiles are narrower than the history's: one normal can't carry t tails. The resampled draws
reproduce the history's extremes and stop at its largest move. Which is better depends on the question: the normal
understates fat tails, the bootstrap can't imagine a move worse than the worst one seen.

## **Minimum Sample: Short History**

The [minimum sample](../README.md#mc6-minimum-sample) rule refuses a class with fewer than `min_sample` returns (30 by default)
instead of fitting a volatility to a handful of them ([`min_sample`](../README.md#min_sample-all-three-simulators)).
Ten sessions of the history hold nine gaps.

In [14]:
# The first ten sessions only: nine gaps, under "independent" so the per-class rule applies
session = index.normalize()
short = values[session.isin(session.unique()[:10])]
try:
    regime_switching_monte_carlo(short, gap_model='independent', n_starts=3, sim_length=5, n_sims=10,
                                 random_state=np.random.default_rng(42), verbose=False)
except ValueError as err:
    print(err)

The gap class has 9 returns, fewer than min_sample=30; about 21 more sessions of history are needed (the history spans 10 sessions). Pass a longer history, or a lower min_sample (0 disables the check)


The message names the class that is short, how many returns it has against `min_sample`, and an estimate
of how much more history would fix it, scaled from how often that class occurs in the history given.